## MODULOS POR CELDA: 

## CONFIG

In [1]:
NOMBRE_APP = "NYC_Taxi_Data"  # Nombre de la spark session
DATASET_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet" 
RUTA_RAW = "./data/raw"  # Ruta del archivo raw
RUTA_TAXI_RAW = "./data/raw/yellow_tripdata_2024-01.parquet"
RUTA_PROCESSED = "./data/processed"  # Ruta del archivo procesado
RUTA_DB = "./data/nyc_taxi.db"  # Ruta de la base SQLite
SEMILLA = 42  # Semilla para la generación de números aleatorios
COLUMNAS_REQUERIDAS = ['tpep_pickup_datetime', 'passenger_count', 'trip_distance', 
                       'PULocationID', 'DOLocationID', 'RatecodeID', 
                       'payment_type', 'fare_amount', 'tip_amount', 
                       'tolls_amount', 'total_amount']
# Lista de columnas que el dataset debe contener para ser considerado válido.
COLUMNAS_IMPUTAR = ['passenger_count', 'RatecodeID'] 
FECHA_INICIO = "2024-01-01"  # Fecha de inicio para filtrar los datos
FECHA_FIN = "2024-02-01"  # Fecha de fin para filtrar los datos
DISTANCIA_MAX_MILLAS = 100  # Distancia máxima en millas para filtrar los datos

DICT_TIPOS = {"passenger_count": "int", "RatecodeID": "int", "payment_type": "int"}
ZONAS_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
RUTA_ZONAS_RAW = "./data/raw/taxi_zone_lookup.csv"

### Resumen del contenido de cada columna del dataset en crudo

In [2]:
'''
Columna	                Qué es	                        Tipo real	       ¿Sirve?

VendorID	            Empresa que provee el taxímetro (1, 2, 6…)Cat	   Poco. Es tecnología, no el viaje

tpep_pickup_datetime	Fecha y hora de subida	        Fecha	           Mucho: de acá salen hora, día, fin de semana

tpep_dropoff_datetime	Fecha y hora de bajada	        Fecha	           Mucho: con la subida da la duración

passenger_count	        Pasajeros    	                Num	               Poco. Casi siempre 1, y no cambia la tarifa

trip_distance	        Distancia en millas 	        Num	               Clave: es lo que más explica el precio

RatecodeID	            Tipo de tarifa: 1 normal,  	    Cat	               Mucho: JFK tiene tarifa fija, cambia todo
                        2 JFK, 3 Newark,
                        4 Nassau/Westchester, 5 negociada, 
                        6 grupal, 99 desconocida

store_and_fwd_flag	    “Y” si el taxi guardó el 	    Cat	                No. Es un dato técnico de transmisión
                        viaje sin conexión y lo envió después

PULocationID	        Zona de subida (1 a 265)	    Cat	                Mucho: geografía, demanda por barrio

DOLocationID	        Zona de bajada (1 a 265)	    Cat	                Mucho

payment_type	        0 flex, 1 tarjeta, 	            Cat	                Para análisis de pago y propinas
                        2 efectivo, 3 sin cargo, 4 disputa

fare_amount	            Tarifa del taxímetro 	        Num	                Candidata a objetivo del modelo
                        (tiempo + distancia), en USD

extra	                Recargos por horario pico 	    Num	                Para el EDA
                        y nocturno

mta_tax	                Impuesto fijo de USD 0,50	    Num	                No: es casi constante

tip_amount	            Propina. Solo se registra 	    Num	                Para análisis de propinas, con cuidado
                        si se pagó con tarjeta: las de efectivo no aparecen

tolls_amount	        Peajes	                        Num	                Para el EDA (indica cruces de puentes o túneles)

improvement_surcharge	Recargo fijo de USD 1	        Num	                No: es constante

total_amount	        Total cobrado 	                Num	                Objetivo alternativo, pero incluye la propina
                        (sin propinas en efectivo)

congestion_surcharge	Recargo por viajes en 	        Num	                Para el EDA (indica zona céntrica)
                        el sur de Manhattan

Airport_fee	            Recargo por subir en los 	    Num	                Para el EDA (indica aeropuerto)
                        aeropuertos LaGuardia o JFK
'''

'\nColumna\t                Qué es\t                        Tipo real\t       ¿Sirve?\n\nVendorID\t            Empresa que provee el taxímetro (1, 2, 6…)Cat\t   Poco. Es tecnología, no el viaje\n\ntpep_pickup_datetime\tFecha y hora de subida\t        Fecha\t           Mucho: de acá salen hora, día, fin de semana\n\ntpep_dropoff_datetime\tFecha y hora de bajada\t        Fecha\t           Mucho: con la subida da la duración\n\npassenger_count\t        Pasajeros    \t                Num\t               Poco. Casi siempre 1, y no cambia la tarifa\n\ntrip_distance\t        Distancia en millas \t        Num\t               Clave: es lo que más explica el precio\n\nRatecodeID\t            Tipo de tarifa: 1 normal,  \t    Cat\t               Mucho: JFK tiene tarifa fija, cambia todo\n                        2 JFK, 3 Newark,\n                        4 Nassau/Westchester, 5 negociada, \n                        6 grupal, 99 desconocida\n\nstore_and_fwd_flag\t    “Y” si el taxi guardó el \t    Cat

### Preguntas que se pueden responder usando agregaciones (EDA, Integrante 2):

* Demanda: ¿a qué hora y qué día hay más viajes? ¿Cambia entre semana y fin de semana?
* Geografía: ¿qué zonas y distritos generan más viajes? ¿Cuáles son los recorridos origen–destino más comunes?
* Precio: ¿cómo se relaciona la tarifa con la distancia? ¿Y con la duración? ¿En qué zonas u horarios es más cara la milla?
* Tránsito: velocidad media por hora y por distrito (la hora pico se ve clarísimo).
* Aeropuertos: cuánto pesan en viajes y en recaudación, y la tarifa fija de JFK.
* Pago: tarjeta vs. efectivo por zona y horario.
* Propinas (solo tarjeta): porcentaje promedio por hora, distrito y distancia.
* Calidad de datos: cuánto se eliminó y por qué (tu parte también es un hallazgo).

### 4. Algoritmos posibles (MLlib, Integrante 3)
Opción	/ Objetivo	 / Tipo	/ Algoritmos MLlib	/ Variables de entrada


A. / Predecir la tarifa /	fare_amount	/ Regresión	LinearRegression (base), RandomForestRegressor, GBTRegressor / distancia, zonas o distritos, hora, día, RatecodeID, es_aeropuerto

B. / Predecir la duración / duracion_min	/ Regresión/	los mismos	distancia, zonas, hora, día

C./ ¿Deja propina?	/ propina > 0 (sí/no)	/ Clasificación	LogisticRegression, RandomForestClassifier	/ distancia, hora, zona, tarifa (solo tarjeta)

D. / Tipos de viaje	/ sin objetivo	/ Clustering KMeans	/ distancia, duración, hora, tarifa

### Recomiendo la opción A:

es la más fácil de explicar (“estimar cuánto va a costar un viaje antes de hacerlo”);
tiene una métrica clara (RMSE en dólares);
se articula naturalmente con el EDA: el EDA muestra que la distancia, la hora y el aeropuerto mueven el precio, y esas variables entran al modelo. Esa es la integración que pide la consigna;
permite comparar un modelo simple (lineal) con uno más potente (Random Forest), lo que suma a las conclusiones.

Cuidado con la “filtración de datos” (data leakage): si el objetivo es fare_amount, no se pueden usar como entradas total_amount, tip_amount, tolls_amount ni extra. Son partes del mismo cobro, se conocen recién al final del viaje y el modelo “haría trampa”. La duración también se conoce al terminar: decidan si el modelo predice antes del viaje (sin duración) o después (con duración), y explíquenlo en el informe.

Qué decidir ahora con el equipo
Objetivo del modelo: A, B, C o D.
Con eso, la lista de variables finales para tu COLUMNAS_FINALES y qué columnas derivadas creás vos.

Si van por la A, tu transformación necesita como mínimo: duracion_min, hora, dia_semana, es_fin_de_semana, es_aeropuerto y el join con zonas. velocidad_mph y duracion_min te sirven también para filtrar viajes absurdos en la limpieza.

In [3]:
# COLUMNAS_REQUIERIDAS describe las columnas que se esperan en el dataset de NYC Taxi. 
# para determianr una prediccion de tarifa
# A continuación se detalla qué representa cada columna:

'''
Columna	                                        Qué es

tpep_pickup_datetime 	                        Base de hora, día y fin de semana; series de tiempo en el EDA
passenger_count	                                Pasajeros (lo carga el chofer, tiene errores)
trip_distance	                                principal, Distancia en millas

PULocationID 
DOLocationID	                                Zona de subida y bajada

RatecodeID	                                    Tarifa normal vs aeropuerto 
payment_type	                                1 tarjeta, 2 efectivo, 3 sin cargo, 4 disputa

fare_amount	                                    OBJETIVO Tarifa base, sin extras

tip_amount, 
tolls_amount, 	                                Propina, peajes y recargos
total_amount	                                Total cobrado
'''

'\nColumna\t                                        Qué es\n\ntpep_pickup_datetime \t                        Base de hora, día y fin de semana; series de tiempo en el EDA\npassenger_count\t                                Pasajeros (lo carga el chofer, tiene errores)\ntrip_distance\t                                principal, Distancia en millas\n\nPULocationID \nDOLocationID\t                                Zona de subida y bajada\n\nRatecodeID\t                                    Tarifa normal vs aeropuerto \npayment_type\t                                1 tarjeta, 2 efectivo, 3 sin cargo, 4 disputa\n\nfare_amount\t                                    OBJETIVO Tarifa base, sin extras\n\ntip_amount, \ntolls_amount, \t                                Propina, peajes y recargos\ntotal_amount\t                                Total cobrado\n'

## SESION_SPARK

In [4]:
from pyspark.sql import SparkSession

def crear_sesion(nombre_app):
    """
    Crea la sesion de spark y la retorna
    """
    sesion = (
        SparkSession.builder
        .appName(nombre_app)                    # nombre de la aplicacion
        .master("local[*]")                     # le dice a spark que corra localmente y que use los nucleos del procesador como executors
        .getOrCreate()                          # crear u obtiene la sesion de spark, si ya hay una sesion abierta la reutiliza, sino crea una nueva
    )                                           # dentro del parentesis = una sola instruccion / syntax error
    sesion.sparkContext.setLogLevel("ERROR")    # setea el nivel de log a ERROR para no mostrar INFO y WARN
    return sesion

## CARGA_DATOS

In [5]:
from importlib.resources import path
import os
from pyspark.shell import spark
import requests

def descargar_dataset(url, ruta_destino): # ruta_destino = RUTA_RAW
    """
    Descarga un archivo desde la URL y lo guarda en disco.
    como Spark solo lee archivos locales, primero se descarga el dataset y luego se lee desde la ruta local.
    si el archivo ya existe, no se descarga nuevamente y se retorna la ruta del archivo existente.

    Parámetros:
    url (str): La URL del dataset a descargar.
    ruta_destino (str): La ruta donde se guardará el archivo descargado.

    Retorna:
    str: La ruta del archivo descargado o existente.
    """
    if os.path.exists(ruta_destino):
        print(f"El archivo ya existe en la ruta: {ruta_destino}. No se descargará nuevamente.")
        return ruta_destino  
    
    # Creo la carpeta donde va a quedar el archivo
    os.makedirs(os.path.dirname(ruta_destino), exist_ok=True)

    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status() # Lanza una excepción si la descarga falla
                                          
        with open(ruta_destino, 'wb') as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024): # Descarga en bloques de 1 MB
                file.write(chunk)
    print(f'Archivo descargado desde {url} y guardado en {ruta_destino}.')
    return ruta_destino  
# ------------------------------------------------------------------------------------------------------------
def leer_dataset(spark, ruta_archivo): # ruta_archivo = RUTA_RAW
    """
    Lee el archivo Parquet de viajes en taxi.
    Parquet guarda los nombres y tipos de las columnas.
    
    Parámetros:
    spark (SparkSession): La sesión de Spark.
    ruta_archivo (str): La ruta del archivo Parquet a leer.
    
    Retorna:
    DataFrame: Un DataFrame de Spark con los datos leídos del archivo Parquet.
    """
    df = spark.read.parquet(ruta_archivo)  # Lee el archivo parquet
    return df
# ------------------------------------------------------------------------------------------------------------
def leer_zonas(spark, ruta_archivo):
    """
    Lee el CSV de zonas de taxi de la TLC.

    Parámetros:
    spark (SparkSession): La sesión de Spark.
    ruta_archivo (str): La ruta del archivo CSV.

    Retorna:
    DataFrame: Tabla de zonas (LocationID, Borough, Zone, service_zone).
    """
    # header=True: la primera fila tiene los nombres de columna
    # inferSchema=True: Spark deduce el tipo de cada columna (en un CSV todo es texto)
    zonas = spark.read.csv(ruta_archivo, header=True, inferSchema=True)
    return zonas
# ------------------------------------------------------------------------------------------------------------
def validar_columnas(dataframe, columnas_requeridas): # columnas_requeridas = COLUMNAS_REQUERIDAS
    """
    Verifica que el DataFrame tenga todas las columnas requeridas. No modifica el DataFrame.

    Parámetros:
    dataframe (DataFrame): El DataFrame de Spark a validar.
    columnas_requeridas (list): Nombres de las columnas que deben estar presentes.

    Retorna:
    None. Lanza ValueError si falta alguna columna.
    """
    faltantes = [c for c in columnas_requeridas if c not in dataframe.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas: {faltantes}")
    print(f"Validación correcta: las {len(columnas_requeridas)} columnas requeridas están presentes.")
# ------------------------------------------------------------------------------------------------------------
    
    #print("Columnas antes:", len(dataframe.columns))
    #dataframe = dataframe.select(columnas_requeridas)
    #print("Columnas después:", len(dataframe.columns))
    #return dataframe
    # en spark los df son inmutables, por lo que se retorna un nuevo df con las columnas requeridas
# ------------------------------------------------------------------------------------------------------------
def cargar_datos(spark):
    """
    Coordina el flujo de carga de los viajes: descarga, lectura y validacion.
    Recibe la sesion como parametro en lugar de crearla: la sesion (y el Driver) se crea una sola vez en main.py
    
    Prametros:
    spark (SparkSession): La sesión de Spark.
    Retorna:
    DataFrame: Los viaje crudos, ya validados.
    """
    ruta = descargar_dataset(DATASET_URL, RUTA_TAXI_RAW)
    df = leer_dataset(spark, ruta)
    validar_columnas(df, COLUMNAS_REQUERIDAS)
    return df
# ------------------------------------------------------------------------------------------------------------
def cargar_zonas(spark):
    """
    Coordina la carga de la tabla de zonas: descarga y lectura.

    Parámetros:
    spark (SparkSession): La sesión de Spark.

    Retorna:
    DataFrame: Tabla de zonas.
    """
    ruta = descargar_dataset(ZONAS_URL, RUTA_ZONAS_RAW)
    return leer_zonas(spark, ruta)


Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /__ / .__/\_,_/_/ /_/\_\   version 4.3.0
      /_/

Using Python version 3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026 10:26:10)
Spark context Web UI available at http://192.168.1.38:4040
Spark context available as 'sc' (master = local[*], app id = local-1793796434279).
SparkSession available as 'spark'.


In [6]:
# dtypes del spark dataset (formato parquet)
'''
Tipo	                Qué es	                                        Columnas
************************************************************************************************************
int	                    Número entero (hasta unos ±2.100 millones)	    VendorID, PULocationID, DOLocationID
bigint	                Número entero “grande” (hasta ±9 trillones).    Es lo mismo que long en printSchema	passenger_count,        RatecodeID,                                     payment_type
double	                Número con decimales	                        distancias y todos los montos
string	                Texto	                                        tore_and_fwd_flag (“Y”/”N”)
timestamp_ntz	        Fecha y hora sin zona horaria 	                subida y bajada
'''

'\nTipo\t                Qué es\t                                        Columnas\n************************************************************************************************************\nint\t                    Número entero (hasta unos ±2.100 millones)\t    VendorID, PULocationID, DOLocationID\nbigint\t                Número entero “grande” (hasta ±9 trillones).    Es lo mismo que long en printSchema\tpassenger_count,        RatecodeID,                                     payment_type\ndouble\t                Número con decimales\t                        distancias y todos los montos\nstring\t                Texto\t                                        tore_and_fwd_flag (“Y”/”N”)\ntimestamp_ntz\t        Fecha y hora sin zona horaria \t                subida y bajada\n'

# Dato para el equipo sobre las columnas 

'VendorID', 'RatecodeID', 'payment_type', 'PULocationID' y 'DOLocationID' son números, pero no son cantidades: son códigos o categorías. 

El 2 de payment_type (efectivo) no es “el doble” del 1 (tarjeta). Esto le importa al Integrante 3: en MLlib esas columnas no se pueden meter como números sin más, se tratan como categorías (por ejemplo, con OneHotEncoder). Y al Integrante 2: no tiene sentido calcular el “promedio de payment_type"

## LIMPIEZA

In [7]:
"""
Módulo: limpieza.py
Descripción: quita los datos erronesos y los nulos del dataset
Autor/a: <Josefina Mercado> — Legajo <número>
Proyecto: 2° Desempeño — Procesamiento de Datos (ISSD)

Entrada: <qué recibe el módulo>
Salida: <qué devuelve o qué archivo genera>
"""

from pyspark.sql.functions import col
#from src.config import (DICT_TIPOS, COLUMNAS_IMPUTAR, FECHA_INICIO, FECHA_FIN,
#                       DISTANCIA_MAX_MILLAS)


def corregir_tipos(df, tipos): 
    """
    Convierte las columnas indicadas al tipo de destino.
    Al leerse desde Parquet el esquema ya está tipado correctamente. 
    Solo se convierten tres columnas de códigos de bigint a int, 
    por consistencia y menor uso de memoria.
    * (ningún valor supera el máximo de int, 2.147.483.647).
    Las fechas se mantuvieron como timestamp_ntz para preservar la hora local de Nueva York.

    Parámetros: 
    df (DataFrame de Spark)  
    tipos (dict) columna : tipo destino.
    
    Retorna: 
    el DataFrame con los tipos corregidos.
    """
    for columna, tipo in tipos.items():
        df = df.withColumn(columna, col(columna).cast(tipo))
    print("- Tipos de datos corregidos:", tipos)
    return df
# ------------------------------------------------------------------------------------------------------------
def eliminar_duplicados(df): 
    """
    Elimina las filas duplicadas de todas las columnas del DataFrame
    
    Parámetros:
    df (DataFrame de Spark): El DataFrame del que se eliminarán los duplicados
    
    Retorna:
    DataFrame: El DataFrame sin duplicados
    """
    df = df.dropDuplicates()
    print("- Duplicados eliminados")
    return df
# ------------------------------------------------------------------------------------------------------------
def tratar_nulos(df, columnas):
    """
    Imputa los nulos de las columnas indicadas con la moda (valor más frecuente).
    Se usa la moda porque son columnas de conteo o de código, donde el promedio no tiene sentido.
    Antes de imputar, marca las filas afectadas en la columna 'nulos_imputados'.
    
    Parámetros:
    df (DataFrame): DataFrame de Spark.
    columnas (list): columnas a imputar.

    Retorna:
    DataFrame: DataFrame sin nulos en esas columnas, con la columna 'nulos_imputados'.
    """
    # 1. Marcar las filas que tienen algún nulo en esas columnas (antes de rellenar)
    condicion = None
    for c in columnas:
        condicion = col(c).isNull() if condicion is None else (condicion | col(c).isNull())
    
    df = df.withColumn("nulos_imputados", condicion)

    # 2. Calcular la moda de cada columna, ignorando los nulos
    print("- Tratamiento de nulos con la moda para las columnas:", columnas)
    modas = {}
    for c in columnas:
        modas[c] = (df.filter(col(c).isNotNull())
                      .groupBy(c).count()
                      .orderBy("count", ascending=False)
                      .first()[0])
        print(f"Moda de {c}: {modas[c]}")

    # 3. Rellenar los nulos con la moda de cada columna
    df = df.fillna(modas)
    print("- Valores nulos tratados")
    return df
# ------------------------------------------------------------------------------------------------------------
def filtrar_invalidos(df):
    """
    Elimina los registros con valores imposibles detectados en el análisis descriptivo:
    - montos de tarifa o total menores o iguales a 0 (reembolsos y anulaciones);
    - distancia igual a 0 o mayor a DISTANCIA_MAX_MILLAS (máximo observado: 312.722 millas);
    - cantidad de pasajeros fuera del rango 1 a 6;
    - RatecodeID fuera de los códigos válidos 1 a 6 (99 = desconocido);
    - viajes con subida fuera del mes analizado;
    - viajes cuya bajada no es posterior a la subida.

    Parámetros:
    df (DataFrame): DataFrame de Spark.

    Retorna:
    DataFrame: El DataFrame solo con registros válidos.
    """
    df = df.filter(
        (col("fare_amount") > 0)
        & (col("total_amount") > 0)
        & (col("trip_distance") > 0)                        # valores mayores a cero
        & (col("trip_distance") <= DISTANCIA_MAX_MILLAS)    # valores menores a DISTANCIA_MAX_MILLAS
        & (col("passenger_count").between(1, 6))
        & (col("RatecodeID").between(1, 6))
        & (col("tpep_pickup_datetime") >= FECHA_INICIO)
        & (col("tpep_pickup_datetime") < FECHA_FIN)         # filtro temporal: solo viajes del mes analizado
        & (col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))
    )
    print("- Registros inválidos eliminados")
    return df
# ------------------------------------------------------------------------------------------------------------
def limpiar_dataset(df): 
    """
    Coordina la limpieza: tipos, nulos, duplicados y valores inválidos.
    Muestra cuántas filas se eliminaron en total.

    Parámetros:
    df (DataFrame): DataFrame crudo de viajes.

    Retorna:
    DataFrame: DataFrame de Spark limpio y listo para transformar.
    """
    filas_iniciales = df.count()

    df = corregir_tipos(df, DICT_TIPOS)
    df = tratar_nulos(df, COLUMNAS_IMPUTAR)
    df = eliminar_duplicados(df)
    df = filtrar_invalidos(df)

    df = df.cache()  # cachear el df limpio para acelerar las siguientes operaciones
    filas_finales = df.count()
    eliminadas = filas_iniciales - filas_finales
    print(f"- Filas antes: {filas_iniciales:,} | después: {filas_finales:,} | "
          f"eliminadas: {eliminadas:,} ({eliminadas / filas_iniciales:.1%})")
    return df

## TRANSFORMACION 

In [8]:
# Columnas a crear con la transformación, para el análisis y modelado
'''
Columna	                Uso
duracion_min	        EDA, y filtro de viajes absurdos
velocidad_mph	        EDA (tránsito), y filtro de errores
hora	                Modelo y EDA
dia_semana	            Modelo y EDA
es_fin_de_semana	    Modelo y EDA
es_aeropuerto	        Modelo y EDA (sale de Airport_fee y RatecodeID)
distrito_subida   
distrito_bajada	        EDA y dashboard (join con zonas). Opcional para el modelo
'''

"""
Módulo: transformacion.py
Descripción: Crea las variables derivadas y deja la tabla lista para el análisis
Autor/a: <Josefina Mercado> — Legajo <número>
Proyecto: 2° Desempeño — Procesamiento de Datos (ISSD)

Entrada: <qué recibe el módulo>
Salida: <qué devuelve o qué archivo genera>
"""
from pyspark.sql import functions as F
from pyspark.sql.functions import col

def agregar_variables_tiempo(df):
    """
    Agrega columnas derivadas de tiempo al DataFrame de Spark.
    
    Parámetros:
    df (DataFrame): El DataFrame de Spark que se va a transformar.
  
    Retorna:
    DataFrame: Un DataFrame de Spark con las variables de tiempo agregadas.
    """
    # Calcular la duración del viaje en minutos y la velocidad en millas por hora
    df = df.withColumn("duracion_min", (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60)
    df = df.withColumn("velocidad_mph", col("trip_distance") / (col("duracion_min") / 60))
    
    # "hora" es columna nueva, F.hour extrae la hora de "tpep_pickup_datetime" en formato 24h (0-23) valor entero
    df = df.withColumn("hora", F.hour("tpep_pickup_datetime"))

    # "dia_semana" es columna nueva, F.dayofweek devuelve un valor entero (1=domingo, 2=lunes, ..., 7=sábado)
    df = df.withColumn("dia_semana", F.dayofweek("tpep_pickup_datetime"))

    # "es_fin_de_semana" es columna nueva, 1 si es sábado o domingo, 0 si es lunes a viernes
    df = df.withColumn("es_fin_de_semana", F.when(F.dayofweek("tpep_pickup_datetime").isin([1, 7]), 1).otherwise(0))
    return df
# ------------------------------------------------------------------------------------------------------------
def agregar_variables_derivadas(df): 
    """
    Parámetros:
    dataframe (DataFrame): El DataFrame de Spark que se va a transformar.
    Retorna:
    DataFrame: Un DataFrame de Spark con las variables derivadas agregadas.
    """
    # Agregar columna "es_aeropuerto": 1 si la ubicación de subida es JFK (2), LaGuardia (1) o Newark (3), 0 en caso contrario
    df = df.withColumn("es_aeropuerto", F.when(F.col("pickup_location_id").isin([1, 2, 3]), 1).otherwise(0))
    # Renombro las columnas de ubicación para que sean más descriptivas
    df = df.withColumnRenamed("PUlocationID", "distrito_subida")
    df = df.withColumnRenamed("DOLocationID", "distrito_bajada")
    return df
# ------------------------------------------------------------------------------------------------------------
def transformar_datos(dataframe, ruta_destino): 
    """
    Parámetros:
    dataframe (DataFrame): El DataFrame de Spark que se va a transformar.
    ruta_destino (str): La ruta donde se guardará el archivo procesado.
    Retorna:
    DataFrame: Un DataFrame de Spark con las transformaciones aplicadas y guardado en la ruta especificada.
    """
    df = agregar_variables_tiempo(dataframe)
    # Aquí se pueden agregar más transformaciones según sea necesario
    return df

# ------------------------------------------------------------------------------------------------------------
def guardar_procesado(dataframe, ruta_destino):     # guardado en parquet
    """
    Parámetros:
    dataframe (DataFrame): El DataFrame de Spark que se va a guardar.
    ruta_destino (str): La ruta donde se guardará el archivo procesado.
  
    Retorna:
    None: Guarda el DataFrame procesado en la ruta especificada.
    """
    # Implementación de la función para guardar el DataFrame procesado en un archivo
    pass  # Reemplazar con la lógica de guardado


## MAIN

In [9]:
#from src.sesion_spark import crear_sesion
#from src import config
#from src.carga_datos import descargar_dataset, leer_dataset, validar_columnas
from pyspark.sql.functions import col, count, when
#from src.config import DICT_TIPOS

# carga de datos
spark = crear_sesion(NOMBRE_APP)
df = cargar_datos(spark)

df.show(5)

# limpieza
df = limpiar_dataset(df)

# cierre de sesion de spark
spark.stop()

Archivo descargado desde https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet y guardado en ./data/raw/yellow_tripdata_2024-01.parquet.
Validación correcta: las 11 columnas requeridas están presentes.
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------